## CrewAI setup with .env\n


In [ ]:
%pip install -q crewai python-dotenv

In [ ]:
from dotenv import load_dotenv
import os
from crewai import LLM

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY was not found in the .env file")

llm = LLM(model="groq/llama-3.1-8b-instant", api_key=api_key)
print("CrewAI is ready. GROQ_API_KEY loaded from .env")

In [ ]:
# Workaround: CrewAI 1.15.x tags every message with a "cache_breakpoint" field
# (an Anthropic-only prompt-caching hint) but never strips it for other providers.
# Groq's API rejects unknown message fields, so disable the tagging.
import crewai.llms.cache as _crewai_cache

_crewai_cache.mark_cache_breakpoint = lambda message: message

## Define a simple Agent, Task, and Crew


In [ ]:
from crewai import Agent, Task, Crew

researcher = Agent(
    role="Research Analyst",
    goal="Find and summarize the latest developments on a given topic",
    backstory="You are an experienced analyst skilled at distilling complex topics into clear summaries.",
    llm=llm,
)

research_task = Task(
    description="Research the current state of {topic} and summarize the key points.",
    expected_output="A concise, well-organized summary of {topic} in 3-5 bullet points.",
    agent=researcher,
)

crew = Crew(
    agents=[researcher],
    tasks=[research_task],
)

result = await crew.kickoff_async(inputs={"topic": "multi-agent AI systems"})
print(result)